In [1]:
# ============================================================================
# CELL 2: LOAD VARIABLES (ضعها في بداية Notebook الجديد)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("📥 LOADING SAVED VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📂 STEP 1: Find save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"

if not save_folder.exists():
    print()
    print("❌ No saved sessions found!")
    print(f"   Expected location: {save_folder}")
    print()
    print("💡 First run CELL 1 in your source notebook to save variables")
    print("=" * 80)
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 📋 STEP 2: List available sessions
    # ═══════════════════════════════════════════════════════════════════════════

    sessions = sorted([d for d in save_folder.iterdir() if d.is_dir()], reverse=True)

    if len(sessions) == 0:
        print()
        print("❌ No sessions found in folder!")
        print(f"   Folder exists but is empty: {save_folder}")
        print()
        print("💡 Run CELL 1 in your source notebook to create a session")
        print("=" * 80)
    else:
        print()
        print(f"📂 Found {len(sessions)} saved session(s):")
        print()
        print("-" * 80)

        # Display available sessions
        for idx, session in enumerate(sessions, 1):
            metadata_path = session / "metadata.pkl"

            if metadata_path.exists():
                try:
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)

                    dt = metadata.get('datetime', metadata.get('timestamp', 'Unknown'))
                    var_count = metadata.get('success_count',
                               len([v for v in metadata.get('saved_variables', {}).values()
                                   if v.get('saved', False)]))

                    print(f"  {idx}. {session.name}")
                    print(f"     Date: {dt}")
                    print(f"     Variables: {var_count}")

                    # Show variable names
                    vars_list = [k for k, v in metadata.get('saved_variables', {}).items()
                                if v.get('saved', False)]
                    if len(vars_list) > 0:
                        print(f"     Contains: {', '.join(vars_list[:5])}")
                        if len(vars_list) > 5:
                            print(f"               ... and {len(vars_list)-5} more")
                    print()
                except:
                    print(f"  {idx}. {session.name} (metadata error)")
                    print()
            else:
                print(f"  {idx}. {session.name} (no metadata)")
                print()

        print("-" * 80)

        # ═══════════════════════════════════════════════════════════════════════════
        # 🎯 STEP 3: Choose session to load
        # ═══════════════════════════════════════════════════════════════════════════

        choice = input("\nEnter session number to load (press Enter for latest): ").strip()

        if choice == '':
            choice = '1'

        try:
            session_idx = int(choice) - 1

            if session_idx < 0 or session_idx >= len(sessions):
                print(f"\n❌ Invalid choice! Must be 1-{len(sessions)}")
                print("=" * 80)
            else:
                selected_session = sessions[session_idx]

                print()
                print("=" * 80)
                print(f"📥 Loading session: {selected_session.name}")
                print("=" * 80)
                print()

                # ═══════════════════════════════════════════════════════════════════════════
                # 📦 STEP 4: Load metadata
                # ═══════════════════════════════════════════════════════════════════════════

                metadata_path = selected_session / "metadata.pkl"

                if metadata_path.exists():
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)
                else:
                    print("⚠️  No metadata found - will try to load all .pkl files")
                    metadata = {'saved_variables': {}}

                # ═══════════════════════════════════════════════════════════════════════════
                # 💾 STEP 5: Load variables
                # ═══════════════════════════════════════════════════════════════════════════

                loaded_count = 0
                failed_count = 0

                saved_vars = metadata.get('saved_variables', {})

                if len(saved_vars) == 0:
                    # No metadata, try all .pkl files
                    pkl_files = list(selected_session.glob("*.pkl"))
                    print(f"Found {len(pkl_files)} .pkl files (excluding metadata)")
                    print()

                    for pkl_file in pkl_files:
                        if pkl_file.name != "metadata.pkl":
                            var_name = pkl_file.stem  # filename without .pkl

                            try:
                                with open(pkl_file, 'rb') as f:
                                    var_value = pickle.load(f)

                                globals()[var_name] = var_value

                                size_kb = pkl_file.stat().st_size / 1024
                                size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"
                                var_type = type(var_value).__name__

                                print(f"✅ {var_name:<25} | {var_type:<15} | {size_str}")
                                loaded_count += 1

                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1
                else:
                    # Use metadata
                    for var_name, var_info in saved_vars.items():
                        if var_info.get('saved', False):
                            try:
                                file_path = selected_session / f"{var_name}.pkl"

                                with open(file_path, 'rb') as f:
                                    var_value = pickle.load(f)

                                # Load into global scope
                                globals()[var_name] = var_value

                                print(f"✅ {var_name:<25} | {var_info.get('type', 'Unknown'):<15} | {var_info.get('size', 'Unknown')}")
                                loaded_count += 1

                            except FileNotFoundError:
                                print(f"❌ {var_name:<25} | FILE NOT FOUND")
                                failed_count += 1
                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1

                # ═══════════════════════════════════════════════════════════════════════════
                # ✅ STEP 6: Summary
                # ═══════════════════════════════════════════════════════════════════════════

                print()
                print("=" * 80)
                print("✅ LOAD COMPLETE!")
                print("=" * 80)
                print(f"✅ Loaded:  {loaded_count} variables")
                print(f"❌ Failed:  {failed_count} variables")
                print("=" * 80)
                print()
                print("💡 Variables are now available in this notebook!")
                print("   Example: print(combined_df.head())")
                print("=" * 80)

        except ValueError:
            print()
            print("❌ Invalid input! Please enter a number")
            print("=" * 80)

📥 LOADING SAVED VARIABLES

📂 Found 20 saved session(s):

--------------------------------------------------------------------------------
  1. session_20260118_151536
     Date: 2026-01-18 15:17:07
     Variables: 7
     Contains: combined_df, combined_df_updated, df, inventory_df, sku_df
               ... and 1 more

  2. session_20260118_151247
     Date: 2026-01-18 15:14:36
     Variables: 5
     Contains: combined_df, combined_df_updated, df, inventory_df

  3. session_20260118_092509
     Date: 2026-01-18 09:25:22
     Variables: 4
     Contains: combined_df, inventory_df, sku_df, section_df

  4. session_20260115_152644
     Date: 2026-01-15 15:26:59
     Variables: 3
     Contains: combined_df, inventory_df, inventory_df_tagropa

  5. session_20260115_121842
     Date: 2026-01-15 12:19:21
     Variables: 2
     Contains: combined_df, inventory_df_dai

  6. session_20260115_110225
     Date: 2026-01-15 11:02:34
     Variables: 1
     Contains: combined_df

  7. session_20260114_

np.float64(209366274.3)

In [2]:
# ============================================================================
# FILE 1: PATTERN & COLOR ANALYSIS - WORKING VERSION
# ============================================================================
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP
# ============================================================================
df = combined_df.copy()  # تأكد إن combined_df موجود عندك

# ============================================================================
# SKU PARSING
# ============================================================================
def parse_sku_code(sku):
    sku_str = str(sku).strip()
    sku_clean = ''.join(c for c in sku_str if c.isdigit())
    collection = sku_clean[:4] if len(sku_clean) >= 4 else sku_clean
    pattern = sku_clean[4:7] if len(sku_clean) >= 7 else ''
    color = sku_clean[7:10] if len(sku_clean) >= 10 else ''
    return {
        'SKU': sku,
        'Collection': collection,
        'Pattern': pattern,
        'Color': color,
        'Collection_Pattern': f"{collection}-{pattern}" if pattern else collection
    }

def add_sku_components(df):
    sku_components = df['SKU'].apply(parse_sku_code)
    components_df = pd.DataFrame(sku_components.tolist())
    df['Collection'] = components_df['Collection']
    df['Pattern'] = components_df['Pattern']
    df['Color'] = components_df['Color']
    df['Collection_Pattern'] = components_df['Collection_Pattern']
    return df

# ============================================================================
# PATTERN ANALYSIS
# ============================================================================
def analyze_patterns(df, min_sales=10):
    sku_level = df.groupby('SKU').agg({
        'bal Qty': 'sum',
        'bal Value': 'sum',
        'CURRENT_STOCK': 'first',
        'OUTSTANDING': 'first',
        'Cost': 'first',
        'Collection_Pattern': 'first',
        'Collection': 'first',
        'Pattern': 'first',
        'Color': 'first'
    }).reset_index()

    pattern_stats = sku_level.groupby('Collection_Pattern').agg({
        'SKU': 'count',
        'bal Qty': 'sum',
        'bal Value': 'sum',
        'CURRENT_STOCK': 'sum',
        'OUTSTANDING': 'sum',
        'Cost': 'mean'
    }).reset_index()

    pattern_stats.columns = ['Pattern', 'SKU_Count', 'Sales_Qty',
                             'Sales_Value', 'Stock', 'OUTSTANDING', 'Avg_Cost']

    pattern_stats['Stock_Value'] = pattern_stats['Stock'] * pattern_stats['Avg_Cost']
    pattern_stats['Avg_Sales_Per_SKU'] = pattern_stats['Sales_Qty'] / pattern_stats['SKU_Count']
    pattern_stats['Stock_Sales_Ratio'] = pattern_stats['Stock'] / pattern_stats['Sales_Qty'].replace(0, 1)
    pattern_stats['Performance_Score'] = pattern_stats['Sales_Value'] - (pattern_stats['Stock_Value'] * 0.5)

    pattern_stats = pattern_stats.sort_values('Performance_Score', ascending=False)
    pattern_filtered = pattern_stats[pattern_stats['Sales_Qty'] >= min_sales].copy()

    best = pattern_filtered.head(10)
    worst = pattern_filtered.tail(10)

    return pattern_stats, best, worst, sku_level

# ============================================================================
# COLOR ANALYSIS
# ============================================================================
def analyze_colors(sku_level):
    color_by_pattern = sku_level.groupby(['Collection_Pattern', 'Color']).agg({
        'SKU': 'count',
        'bal Qty': 'sum',
        'bal Value': 'sum',
        'CURRENT_STOCK': 'sum',
        'OUTSTANDING': 'sum',
        'Cost': 'mean'
    }).reset_index()

    color_by_pattern.columns = ['Pattern', 'Color', 'SKU_Count', 'Sales_Qty',
                                'Sales_Value', 'Stock', 'OUTSTANDING', 'Avg_Cost']
    color_by_pattern['Performance_Score'] = color_by_pattern['Sales_Value'] - (color_by_pattern['Stock'] * color_by_pattern['Avg_Cost'] * 0.5)
    color_by_pattern = color_by_pattern.sort_values('Performance_Score', ascending=False)

    color_overall = sku_level.groupby('Color').agg({
        'SKU': 'count',
        'bal Qty': 'sum',
        'bal Value': 'sum',
        'CURRENT_STOCK': 'sum',
        'OUTSTANDING': 'sum'
    }).reset_index()
    color_overall.columns = ['Color', 'SKU_Count', 'Sales_Qty', 'Sales_Value', 'Stock', 'OUTSTANDING']
    color_overall = color_overall.sort_values('Sales_Value', ascending=False)

    best_colors = color_overall.head(10)
    worst_colors = color_overall.tail(10)

    return color_by_pattern, color_overall, best_colors, worst_colors

# ============================================================================
# EXPORT RESULTS (FIXED PATH)
# ============================================================================
def export_results(pattern_stats, best, worst, color_by_pattern, color_overall,
                  best_colors, worst_colors, folder="analysis_results"):
    try:
        os.makedirs(folder, exist_ok=True)
    except Exception as e:
        print(f"⚠️ Could not create folder: {e}")
        return None

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(folder, f'PATTERN_COLOR_ANALYSIS_{ts}.xlsx')

    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            pattern_stats.to_excel(writer, sheet_name='All Patterns', index=False)
            best.to_excel(writer, sheet_name='Best Patterns', index=False)
            worst.to_excel(writer, sheet_name='Worst Patterns', index=False)
            color_by_pattern.to_excel(writer, sheet_name='Colors by Pattern', index=False)
            color_overall.to_excel(writer, sheet_name='Colors Overall', index=False)
            best_colors.to_excel(writer, sheet_name='Best Colors', index=False)
            worst_colors.to_excel(writer, sheet_name='Worst Colors', index=False)

        print(f"✓ Successfully exported to:")
        print(f"  {filename}")
        return filename

    except PermissionError:
        print("✗ Permission denied! File might be open in Excel or you don't have write access.")
        print(f"  Try closing Excel or changing folder → {folder}")
        return None
    except Exception as e:
        print(f"✗ Export failed: {type(e).__name__}")
        print(f"  → {str(e)}")
        return None

# ============================================================================
# MAIN FUNCTION
# ============================================================================
def run_pattern_analysis(combined_df, min_sales=10, folder="."):
    df_enhanced = add_sku_components(combined_df)
    pattern_stats, best, worst, sku_level = analyze_patterns(df_enhanced, min_sales=min_sales)
    color_by_pattern, color_overall, best_colors, worst_colors = analyze_colors(sku_level)
    filename = export_results(pattern_stats, best, worst, color_by_pattern, color_overall, best_colors, worst_colors, folder)
    return {
        'df_updated': df_enhanced,
        'sku_level': sku_level,
        'patterns': pattern_stats,
        'best': best,
        'worst': worst,
        'colors': color_by_pattern,
        'color_overall': color_overall,
        'export_file': filename
    }


# ============================================================================
# USAGE
# ============================================================================
results = run_pattern_analysis(combined_df)
combined_df_updated = results['df_updated']
print(results['patterns'].head())


✓ Successfully exported to:
  .\PATTERN_COLOR_ANALYSIS_20260118_150621.xlsx
       Pattern  SKU_Count    Sales_Qty   Sales_Value      Stock  OUTSTANDING  \
6575  4689-010        151  1482153.260  2.045506e+07  143484.45          0.0   
6691  4738-010         37   448506.950  1.846170e+07   21868.82        800.0   
7800  5031-010        150  1348358.610  1.488515e+07  106497.81       4364.4   
6063  4424-010         80  1162407.787  1.272737e+07  105460.98       1600.0   
6074  4430-010         88   200773.060  1.050076e+07   20752.12       8118.2   

      Avg_Cost  Stock_Value  Avg_Sales_Per_SKU  Stock_Sales_Ratio  \
6575       7.0   1004391.15        9815.584503           0.096808   
6691      17.0    371769.94       12121.809459           0.048759   
7800       4.0    425991.24        8989.057400           0.078983   
6063       5.0    527304.90       14530.097337           0.090726   
6074      18.0    373538.16        2281.512045           0.103361   

      Performance_Score  
65

In [3]:
results['df_updated'].head()

,Route,Route Name,cliroute,Region,Region Name,Gen,Section,Section Name,bal Value,bal Qty,...,Unit_Price,Total_Cost,Total_Profit,Profit_Margin_%,Sales_Type,SKU_CLEAN,Collection,Pattern,Color,Collection_Pattern
0,1,مدينة الرياض,1,ZZ,الرياض / الوسطى,T,3813,جاكار امبيوريو,513.510,51.3,...,10.009942,923.4,-409.890,-79.821230,FAMILY,3813010013,3813,010,013,3813-010
1,2,جدة,1,D,الغربية,T,N116,,443.740,32.0,...,13.866875,NaN,NaN,NaN,FAMILY,N11612,1161,,,1161
2,1,مدينة الرياض,1,B,الرياض / الوسطى,V,3884,أورقانزا تايتان,316.800,12.0,...,26.400000,180.0,136.800,43.181818,FAMILY,3884010016,3884,010,016,3884-010
3,2,مدينة مكه,2,C,الغربية,T,4883,قطن استر بلس,112.060,4.5,...,24.902222,54.0,58.060,51.811530,FAMILY,4883012012,4883,012,012,4883-012
4,2,رابغ,5,0,الغربية,V,4410,دانتيل بامبو,-133.488,-3.0,...,44.496000,-24.0,-109.488,82.020856,FAMILY,4410010012,4410,010,012,4410-010


## Pattern Analysis Dashboard

In [3]:
# ============================================================================
# FILE 1: PATTERN & COLOR DASHBOARD - COMPLETE & READY
# ✅ Fixed export to Desktop/pattern_color_analysis/
# ============================================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.io as pio
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PDF Export
try:
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak, Image
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors as rl_colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    from reportlab.pdfgen import canvas
    import io
    from PIL import Image as PILImage
    PDF_OK = True
except:
    PDF_OK = False
    print("⚠️  PDF export not available (install: pip install reportlab pillow)")

class PatternColorDashboard:
    """Professional Pattern & Color Dashboard"""

    def __init__(self, df):
        print("🔧 Initializing Professional Dashboard...")

        if 'OUTSTANDING' not in df.columns:
            print("⚠️  No OUTSTANDING column - adding with zeros")
            df = df.copy()
            df['OUTSTANDING'] = 0
        else:
            print(f"✅ OUTSTANDING: {df['OUTSTANDING'].sum():,.0f} units")

        self.df = df.copy()
        self.df = self.df.dropna(subset=['Date'])

        required = ['Collection', 'Pattern', 'Color', 'Collection_Pattern']
        missing = [c for c in required if c not in self.df.columns]

        if missing:
            print(f"⚠️  Missing columns: {missing} - parsing SKUs...")
            self._parse_sku()
        else:
            print("✅ SKU components found")

        self.df['Year'] = self.df['Date'].dt.year.astype(int)
        self.years = sorted(self.df['Year'].unique())

        self.has_profit = 'Total_Profit' in self.df.columns
        self.has_stock = 'CURRENT_STOCK' in self.df.columns
        self.has_OUTSTANDING = 'OUTSTANDING' in self.df.columns
        self.has_entry_date = 'LAST_ENTRY_DATE' in self.df.columns

        print("⚡ Pre-computing stock snapshot...")

        cols = ['SKU', 'Collection', 'Pattern', 'Color', 'Collection_Pattern', 'CURRENT_STOCK']
        if self.has_OUTSTANDING:
            cols.append('OUTSTANDING')
        if self.has_entry_date:
            cols.append('LAST_ENTRY_DATE')

        self.stock_snap = self.df.sort_values('Date').groupby('SKU').last().reset_index()[cols]

        if not self.has_stock:
            self.stock_snap['CURRENT_STOCK'] = 0
        if not self.has_OUTSTANDING:
            self.stock_snap['OUTSTANDING'] = 0
        if not self.has_entry_date:
            self.stock_snap['LAST_ENTRY_DATE'] = pd.NaT

        self.stock_snap.fillna(0, inplace=True)

        self.curr_coll = None
        self.curr_years = None
        self.results = {}

        self._widgets()

        print(f"✅ Dashboard Ready!")
        print(f"   Collections: {self.df['Collection'].nunique()}")
        print(f"   Years: {self.years}")
        print(f"   Total Stock: {self.stock_snap['CURRENT_STOCK'].sum():,.0f}")
        print(f"   Total OUTSTANDING: {self.stock_snap['OUTSTANDING'].sum():,.0f}")

    def _parse_sku(self):
        """Parse SKU if components missing"""
        def parse(sku):
            s = ''.join(c for c in str(sku) if c.isdigit())
            if len(s) >= 10:
                return {'Collection': s[:4], 'Pattern': s[4:7], 'Color': s[7:10],
                       'Collection_Pattern': f"{s[:4]}-{s[4:7]}"}
            return {'Collection': s[:4], 'Pattern': '', 'Color': '', 'Collection_Pattern': s[:4]}

        c = pd.DataFrame(self.df['SKU'].apply(parse).tolist())
        for col in c.columns:
            self.df[col] = c[col]

    def _widgets(self):
        """Create UI widgets"""
        colls = sorted([c for c in self.df['Collection'].unique() if c])

        self.coll_box = widgets.Combobox(
            options=colls, value=colls[0] if colls else '',
            ensure_option=False,
            description='Collection:',
            layout=widgets.Layout(width='300px')
        )

        self.year_sel = widgets.SelectMultiple(
            options=[(str(y), y) for y in self.years],
            value=[self.years[-1]],
            description='Years:',
            layout=widgets.Layout(width='300px', height='100px')
        )

        self.btn_analyze = widgets.Button(
            description='🔍 Analyze',
            button_style='primary',
            layout=widgets.Layout(width='300px', height='50px')
        )
        self.btn_analyze.on_click(self._do_analyze)

        self.btn_pdf = widgets.Button(
            description='📄 Export PDF',
            button_style='success',
            layout=widgets.Layout(width='300px', height='50px'),
            disabled=True
        )
        self.btn_pdf.on_click(self._do_pdf)

        self.out = widgets.Output()

    def _do_analyze(self, b):
        """Handle analyze button"""
        with self.out:
            clear_output(wait=True)
            c, y = self.coll_box.value, list(self.year_sel.value)
            if not c or not y:
                print("⚠️  Select collection & years")
                return
            self.curr_coll, self.curr_years = c, y
            self._analyze(c, y)
            if PDF_OK:
                self.btn_pdf.disabled = False

    def _do_pdf(self, b):
        """Handle PDF export"""
        if not PDF_OK:
            print("❌ Install: pip install reportlab pillow")
            return
        with self.out:
            print("\n📄 Generating Professional PDF Report...")
            fn = self._export_pdf()
            if fn:
                print(f"✅ Saved: {fn}")

    def _calculate_advanced_metrics(self, sku_full, df_all):
        """Calculate advanced business metrics for each SKU"""

        today = pd.Timestamp.now()
        latest_year = self.years[-1]

        advanced_metrics = []

        for _, row in sku_full.iterrows():
            sku = row['SKU']
            sku_data = df_all[df_all['SKU'] == sku].copy()

            if len(sku_data) == 0:
                continue

            qty = row['bal Qty']
            avg_price = row['bal Value'] / qty if qty > 0 else 0

            if self.has_entry_date:
                entry_date = sku_data['LAST_ENTRY_DATE'].iloc[0]
                if pd.notna(entry_date):
                    stock_age = (today - pd.to_datetime(entry_date)).days
                else:
                    stock_age = 0
            else:
                stock_age = 0

            sales_days_all = sku_data[sku_data['bal Qty'] > 0]['Date'].nunique()

            sku_latest = sku_data[sku_data['Year'] == latest_year]
            sales_days_latest = sku_latest[sku_latest['bal Qty'] > 0]['Date'].nunique()

            advanced_metrics.append({
                'SKU': sku,
                'Qty': qty,
                'Avg_Price': avg_price,
                'Stock_Age_Days': stock_age,
                'Sales_Days_All': sales_days_all,
                'Sales_Days_Latest': sales_days_latest
            })

        return pd.DataFrame(advanced_metrics)

    def _analyze(self, coll, years):
        """Complete analysis"""

        df_filtered = self.df[
            (self.df['Collection'] == coll) &
            (self.df['Year'].isin(years))
        ].copy()

        df_all = self.df[self.df['Collection'] == coll].copy()

        if len(df_filtered) == 0:
            print(f"❌ No data for Collection {coll} in {years}")
            return

        sku_sales_selected = df_filtered.groupby(['SKU', 'Collection_Pattern', 'Color']).agg({
            'bal Value': 'sum',
            'bal Qty': 'sum',
            'Total_Profit': 'sum' if self.has_profit else lambda x: 0
        }).reset_index()

        sku_sales_all = df_all.groupby(['SKU', 'Collection_Pattern', 'Color']).agg({
            'bal Value': 'sum',
            'bal Qty': 'sum',
            'Total_Profit': 'sum' if self.has_profit else lambda x: 0
        }).reset_index()

        stock_f = self.stock_snap[self.stock_snap['Collection'] == coll].copy()
        sku_full = sku_sales_all.merge(
            stock_f[['SKU', 'CURRENT_STOCK', 'OUTSTANDING', 'LAST_ENTRY_DATE']],
            on='SKU', how='left'
        ).fillna(0)

        print("📊 Calculating advanced metrics...")
        advanced = self._calculate_advanced_metrics(sku_full, df_all)

        sku_full = sku_full.merge(advanced, on='SKU', how='left')

        pattern_sales = sku_sales_selected.groupby('Collection_Pattern').agg({
            'bal Value': 'sum',
            'bal Qty': 'sum',
            'Total_Profit': 'sum',
            'SKU': 'nunique'
        }).reset_index()

        pattern_stock = stock_f.groupby('Collection_Pattern').agg({
            'CURRENT_STOCK': 'sum',
            'OUTSTANDING': 'sum'
        }).reset_index()

        pat = pattern_sales.merge(pattern_stock, on='Collection_Pattern', how='left').fillna(0)
        pat['Margin_%'] = (pat['Total_Profit'] / pat['bal Value'] * 100).fillna(0)
        pat = pat.sort_values('bal Value', ascending=False)

        self.results = {
            'sku_sel': sku_sales_selected,
            'sku_full': sku_full,
            'pat': pat,
            'stock': stock_f,
            'df_all': df_all
        }

        self._show_header(coll, years)
        self._show_metrics(sku_sales_selected, stock_f)
        self._show_patterns(pat)
        self._show_skus(sku_full)
        self._show_charts(pat)

    def _show_header(self, c, y):
        """Display header"""
        h = f"""<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 35px; border-radius: 15px; margin-bottom: 30px; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
        <h1 style='color: #FFF; margin: 0; font-size: 40px; text-align: center; font-weight: 900;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.3);'>🎨 Collection {c}</h1>
        <p style='color: #FFF; margin: 15px 0 0; font-size: 20px; text-align: center; font-weight: 600;'>
        📅 {", ".join(map(str, y))}</p></div>"""
        display(HTML(h))

    def _show_metrics(self, ss, st):
        """Display metrics cards"""
        tot_s = ss['bal Value'].sum()
        tot_q = ss['bal Qty'].sum()
        tot_p = ss['Total_Profit'].sum()
        marg = (tot_p / tot_s * 100) if tot_s else 0
        stock = st['CURRENT_STOCK'].sum()
        outst = st['OUTSTANDING'].sum()
        n_sku = ss['SKU'].nunique()

        h = "<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 20px; margin-bottom: 30px;'>"

        cards = [
            ('💰 Total Sales', f'{tot_s:,.0f}', 'SAR', '#667eea', '#764ba2'),
            ('📦 Quantity', f'{tot_q:,.0f}', 'Units', '#f093fb', '#f5576c'),
            ('🚚 OUTSTANDING', f'{outst:,.0f}', 'Units', '#4facfe', '#00f2fe'),
            ('📊 Stock', f'{stock:,.0f}', 'Units', '#43e97b', '#38f9d7'),
            ('💎 Profit', f'{tot_p:,.0f}', f'Margin: {marg:.1f}%', '#fa709a', '#fee140'),
            ('📋 Products', f'{n_sku} SKUs', 'Active', '#a8edea', '#fed6e3')
        ]

        for title, value, subtitle, c1, c2 in cards:
            h += f"""<div style='background: linear-gradient(135deg, {c1} 0%, {c2} 100%);
            padding: 28px; border-radius: 12px; color: white; box-shadow: 0 6px 20px rgba(0,0,0,0.25);'>
            <div style='font-size: 15px; font-weight: 700; margin-bottom: 10px;'>{title}</div>
            <div style='font-size: 36px; font-weight: 900; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);'>{value}</div>
            <div style='font-size: 13px; font-weight: 600; margin-top: 6px;'>{subtitle}</div></div>"""

        h += "</div>"
        display(HTML(h))

    def _show_patterns(self, df):
        """Display pattern table"""
        h = """<div style='background: white; padding: 28px; border-radius: 12px; margin-bottom: 30px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.15); border: 2px solid #e0e0e0;'>
        <h2 style='margin-top: 0; color: #000; font-weight: 900; font-size: 24px;'>🎨 Pattern Analysis</h2>
        <table style='width: 100%; border-collapse: collapse; font-size: 14px;'>
        <thead><tr style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white;'>
        <th style='padding: 14px; text-align: left; font-weight: 700;'>Pattern</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>Sales</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>Qty</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>Stock</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>OUTSTANDING</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>Profit</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>Margin%</th>
        <th style='padding: 14px; text-align: right; font-weight: 700;'>SKUs</th>
        </tr></thead><tbody>"""

        for i, r in df.iterrows():
            bg = '#f8f9fa' if i % 2 == 0 else 'white'
            pc = '#00aa00' if r['Total_Profit'] > 0 else '#cc0000'
            h += f"""<tr style='background: {bg};'>
            <td style='padding: 12px; font-weight: 700; color: #000;'>{r['Collection_Pattern']}</td>
            <td style='padding: 12px; text-align: right; color: #000; font-weight: 600;'>{r['bal Value']:,.0f}</td>
            <td style='padding: 12px; text-align: right; color: #000; font-weight: 600;'>{r['bal Qty']:,.0f}</td>
            <td style='padding: 12px; text-align: right; color: #000; font-weight: 600;'>{r['CURRENT_STOCK']:,.0f}</td>
            <td style='padding: 12px; text-align: right; color: #0066cc; font-weight: 700;'>{r['OUTSTANDING']:,.0f}</td>
            <td style='padding: 12px; text-align: right; color: {pc}; font-weight: 700;'>{r['Total_Profit']:,.0f}</td>
            <td style='padding: 12px; text-align: right; color: #000; font-weight: 600;'>{r['Margin_%']:.1f}%</td>
            <td style='padding: 12px; text-align: right; color: #000; font-weight: 600;'>{r['SKU']}</td>
            </tr>"""

        h += "</tbody></table></div>"
        display(HTML(h))

    def _show_skus(self, df):
        """Display top SKUs with advanced metrics"""
        top = df.sort_values('bal Value', ascending=False).head(20)

        h = """<div style='background: white; padding: 28px; border-radius: 12px; margin-bottom: 30px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.15); border: 2px solid #e0e0e0;'>
        <h2 style='margin-top: 0; color: #000; font-weight: 900; font-size: 24px;'>🏆 Top 20 SKUs</h2>
        <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
        <thead><tr style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); color: white;'>
        <th style='padding: 10px; font-weight: 700;'>Rank</th>
        <th style='padding: 10px; text-align: left; font-weight: 700;'>SKU</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Sales</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Qty</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Avg Price</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Stock</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Out.</th>
        <th style='padding: 10px; text-align: right; font-weight: 700;'>Age</th>
        </tr></thead><tbody>"""

        medals = {1: "🥇", 2: "🥈", 3: "🥉"}
        for rk, (_, r) in enumerate(top.iterrows(), 1):
            bg = '#f8f9fa' if rk % 2 == 0 else 'white'
            m = medals.get(rk, "")
            h += f"""<tr style='background: {bg};'>
            <td style='padding: 8px; font-weight: 700; color: #000;'>{m} {rk}</td>
            <td style='padding: 8px; font-family: monospace; color: #000; font-weight: 600;'>{r['SKU']}</td>
            <td style='padding: 8px; text-align: right; font-weight: 700; color: #000;'>{r['bal Value']:,.0f}</td>
            <td style='padding: 8px; text-align: right; color: #000; font-weight: 600;'>{r.get('Qty', r['bal Qty']):,.0f}</td>
            <td style='padding: 8px; text-align: right; color: #000; font-weight: 600;'>{r.get('Avg_Price', 0):,.2f}</td>
            <td style='padding: 8px; text-align: right; color: #000; font-weight: 600;'>{r['CURRENT_STOCK']:,.0f}</td>
            <td style='padding: 8px; text-align: right; color: #0066cc; font-weight: 700;'>{r['OUTSTANDING']:,.0f}</td>
            <td style='padding: 8px; text-align: right; color: #666; font-weight: 600;'>{r.get('Stock_Age_Days', 0):.0f}d</td>
            </tr>"""

        h += "</tbody></table></div>"
        display(HTML(h))

    def _show_charts(self, df):
        """Display charts"""
        fig1 = go.Figure(go.Bar(
            x=df['Collection_Pattern'], y=df['bal Value'], marker_color='#667eea',
            text=df['bal Value'].apply(lambda x: f'{x:,.0f}'), textposition='outside'
        ))
        fig1.update_layout(title='Pattern Sales', xaxis_title='Pattern', yaxis_title='Sales (SAR)',
                          template='plotly_white', height=500)
        display(fig1)

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(name='Stock', x=df['Collection_Pattern'], y=df['CURRENT_STOCK'], marker_color='#43e97b'))
        fig2.add_trace(go.Bar(name='OUTSTANDING', x=df['Collection_Pattern'], y=df['OUTSTANDING'], marker_color='#4facfe'))
        fig2.update_layout(title='Stock vs OUTSTANDING', barmode='group', template='plotly_white', height=500)
        display(fig2)

    def _create_chart_image(self, fig, width=6, height=4):
        """Convert Plotly chart to PIL Image for PDF"""
        try:
            img_bytes = pio.to_image(fig, format='png', width=width*100, height=height*100)
            img = PILImage.open(io.BytesIO(img_bytes))

            img_io = io.BytesIO()
            img.save(img_io, format='PNG')
            img_io.seek(0)

            return Image(img_io, width=width*inch, height=height*inch)
        except:
            return None

    def _export_pdf(self):
        """✅ FIXED: Export to Desktop/pattern_color_analysis/"""

        # ✅ Create organized folder on Desktop
        desktop = Path.home() / "Desktop"
        main_folder = desktop / "pattern_color_analysis"
        main_folder.mkdir(exist_ok=True)

        # Create timestamped subfolder
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        session_folder = main_folder / ts
        session_folder.mkdir(exist_ok=True)

        # Create filename
        fn = session_folder / f"Collection_{self.curr_coll}_Professional_Report_{ts}.pdf"

        doc = SimpleDocTemplate(str(fn), pagesize=A4,
                               topMargin=0.75*inch, bottomMargin=0.75*inch,
                               leftMargin=0.5*inch, rightMargin=0.5*inch)
        story = []
        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=28,
            textColor=rl_colors.HexColor('#667eea'),
            alignment=TA_CENTER,
            spaceAfter=6,
            fontName='Helvetica-Bold'
        )

        subtitle_style = ParagraphStyle(
            'CustomSubtitle',
            parent=styles['Normal'],
            fontSize=12,
            textColor=rl_colors.HexColor('#666666'),
            alignment=TA_CENTER,
            spaceAfter=20
        )

        heading_style = ParagraphStyle(
            'CustomHeading',
            parent=styles['Heading2'],
            fontSize=16,
            textColor=rl_colors.HexColor('#667eea'),
            spaceBefore=20,
            spaceAfter=12,
            fontName='Helvetica-Bold'
        )

        story.append(Spacer(1, 1*inch))
        story.append(Paragraph(f"Collection {self.curr_coll}", title_style))
        story.append(Paragraph(
            f"Pattern & Color Analysis Report",
            subtitle_style
        ))
        story.append(Paragraph(
            f"Period: {', '.join(map(str, self.curr_years))} | Generated: {datetime.now().strftime('%d %B %Y, %H:%M')}",
            subtitle_style
        ))

        story.append(Spacer(1, 0.5*inch))

        story.append(Paragraph("📊 Executive Summary", heading_style))

        ss = self.results['sku_sel']
        st = self.results['stock']

        summary_data = [
            ['Metric', 'Value', 'Metric', 'Value'],
            ['Total Sales', f"{ss['bal Value'].sum():,.0f} SAR", 'Total Quantity', f"{ss['bal Qty'].sum():,.0f} units"],
            ['Total Profit', f"{ss['Total_Profit'].sum():,.0f} SAR", 'Avg Margin', f"{(ss['Total_Profit'].sum() / ss['bal Value'].sum() * 100):.1f}%"],
            ['Current Stock', f"{st['CURRENT_STOCK'].sum():,.0f} units", 'OUTSTANDING PO', f"{st['OUTSTANDING'].sum():,.0f} units"],
            ['Active SKUs', f"{ss['SKU'].nunique():,}", 'Patterns', f"{self.results['pat'].shape[0]:,}"]
        ]

        summary_table = Table(summary_data, colWidths=[2*inch, 1.5*inch, 2*inch, 1.5*inch])
        summary_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#667eea')),
            ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 11),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('BACKGROUND', (0, 1), (-1, -1), rl_colors.beige),
            ('GRID', (0, 0), (-1, -1), 1.5, rl_colors.grey),
            ('FONTNAME', (0, 1), (0, -1), 'Helvetica-Bold'),
            ('FONTNAME', (2, 1), (2, -1), 'Helvetica-Bold'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey])
        ]))
        story.append(summary_table)

        story.append(PageBreak())
        story.append(Paragraph("🎨 Pattern Analysis", heading_style))

        pat = self.results['pat'].head(30)

        pat_data = [['Pattern', 'Sales\n(SAR)', 'Qty', 'Stock', 'Out.', 'Profit\n(SAR)', 'Margin\n%', 'SKUs', 'Out/Stock\n%']]
        for _, r in pat.iterrows():
            out_stock_ratio = f"{(r['OUTSTANDING'] / r['CURRENT_STOCK']*100):.0f}" if r['CURRENT_STOCK'] > 0 else "N/A"
            pat_data.append([
                r['Collection_Pattern'],
                f"{r['bal Value']:,.0f}",
                f"{r['bal Qty']:,.0f}",
                f"{r['CURRENT_STOCK']:,.0f}",
                f"{r['OUTSTANDING']:,.0f}",
                f"{r['Total_Profit']:,.0f}",
                f"{r['Margin_%']:.1f}",
                str(int(r['SKU'])),
                out_stock_ratio
            ])

        pat_table = Table(pat_data, repeatRows=1,
                         colWidths=[0.9*inch, 0.8*inch, 0.6*inch, 0.6*inch, 0.6*inch, 0.8*inch, 0.6*inch, 0.5*inch, 0.6*inch])

        style_cmds = [
            ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor('#667eea')),
            ('TEXTCOLOR', (0,0), (-1,0), rl_colors.whitesmoke),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,0), 9),
            ('ALIGN', (1,1), (-1,-1), 'CENTER'),
            ('ALIGN', (0,0), (0,-1), 'LEFT'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('GRID', (0,0), (-1,-1), 0.5, rl_colors.grey),
            ('FONTSIZE', (0,1), (-1,-1), 8),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [rl_colors.white, rl_colors.lightgrey])
        ]

        for i, row in enumerate(pat_data[1:], 1):
            try:
                stock = float(row[3].replace(',',''))
                OUTSTANDING = float(row[4].replace(',',''))

                if stock == 0:
                    if OUTSTANDING > 0:
                        style_cmds.append(('BACKGROUND', (3,i), (3,i), rl_colors.Color(0.7, 0.9, 0.7)))
                    else:
                        style_cmds.append(('BACKGROUND', (3,i), (3,i), rl_colors.Color(1, 0.7, 0.7)))

                elif stock < 10:
                    style_cmds.append(('BACKGROUND', (3,i), (3,i), rl_colors.Color(1, 1, 0.7)))
            except:
                pass

        pat_table.setStyle(TableStyle(style_cmds))
        story.append(pat_table)

        story.append(Spacer(1, 0.3*inch))
        story.append(Paragraph("📈 Sales by Pattern", ParagraphStyle('ChartTitle', fontSize=12, textColor=rl_colors.HexColor('#666666'))))

        fig_pattern = go.Figure(go.Bar(
            x=pat['Collection_Pattern'].head(15),
            y=pat['bal Value'].head(15),
            marker_color='#667eea',
            text=pat['bal Value'].head(15).apply(lambda x: f'{x:,.0f}'),
            textposition='outside'
        ))
        fig_pattern.update_layout(
            title='',
            xaxis_title='Pattern',
            yaxis_title='Sales (SAR)',
            template='plotly_white',
            height=350,
            margin=dict(l=50, r=50, t=30, b=80),
            xaxis_tickangle=-45
        )

        chart_img = self._create_chart_image(fig_pattern, width=7, height=3.5)
        if chart_img:
            story.append(chart_img)

        story.append(PageBreak())
        story.append(Paragraph("📦 Stock & OUTSTANDING Analysis", heading_style))

        fig_stock = go.Figure()
        fig_stock.add_trace(go.Bar(
            name='Current Stock',
            x=pat['Collection_Pattern'].head(15),
            y=pat['CURRENT_STOCK'].head(15),
            marker_color='#43e97b'
        ))
        fig_stock.add_trace(go.Bar(
            name='OUTSTANDING PO',
            x=pat['Collection_Pattern'].head(15),
            y=pat['OUTSTANDING'].head(15),
            marker_color='#4facfe'
        ))
        fig_stock.update_layout(
            title='',
            xaxis_title='Pattern',
            yaxis_title='Units',
            template='plotly_white',
            barmode='group',
            height=400,
            margin=dict(l=50, r=50, t=30, b=80),
            xaxis_tickangle=-45,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )

        chart_img2 = self._create_chart_image(fig_stock, width=7, height=4)
        if chart_img2:
            story.append(chart_img2)

        story.append(PageBreak())
        story.append(Paragraph("📋 Complete SKU Analysis", heading_style))

        skus = self.results['sku_full'].sort_values('bal Value', ascending=False)

        skus_per_page = 22

        for pg, i in enumerate(range(0, len(skus), skus_per_page)):
            if pg > 0:
                story.append(PageBreak())
                story.append(Paragraph(f"📋 Complete SKU Analysis (cont'd - Page {pg+1})", heading_style))

            page_skus = skus.iloc[i:i+skus_per_page]

            sku_data = [['#', 'SKU', 'Sales', 'Qty', 'Avg\nPrice', 'Stock', 'Out.', 'Age\n(d)', 'Sales\nDays\n(All)', 'Sales\nDays\n(2025)']]

            for rk, (_, r) in enumerate(page_skus.iterrows(), start=i+1):
                sku_data.append([
                    str(rk),
                    str(r['SKU']),
                    f"{r['bal Value']:,.0f}",
                    f"{r.get('Qty', r['bal Qty']):,.0f}",
                    f"{r.get('Avg_Price', 0):,.2f}",
                    f"{r['CURRENT_STOCK']:,.0f}",
                    f"{r['OUTSTANDING']:,.0f}",
                    f"{r.get('Stock_Age_Days', 0):.0f}",
                    f"{r.get('Sales_Days_All', 0):.0f}",
                    f"{r.get('Sales_Days_Latest', 0):.0f}"
                ])

            sku_table = Table(sku_data, repeatRows=1,
                            colWidths=[0.3*inch, 1.4*inch, 0.8*inch, 0.6*inch, 0.7*inch, 0.6*inch, 0.6*inch, 0.5*inch, 0.6*inch, 0.6*inch])

            style_cmds = [
                ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor('#f093fb')),
                ('TEXTCOLOR', (0,0), (-1,0), rl_colors.whitesmoke),
                ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
                ('FONTSIZE', (0,0), (-1,0), 8),
                ('ALIGN', (0,0), (-1,-1), 'CENTER'),
                ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
                ('GRID', (0,0), (-1,-1), 0.5, rl_colors.grey),
                ('FONTSIZE', (0,1), (-1,-1), 7),
                ('ROWBACKGROUNDS', (0,1), (-1,-1), [rl_colors.white, rl_colors.lightgrey])
            ]

            for row_idx, row in enumerate(sku_data[1:], 1):
                try:
                    stock = float(row[5].replace(',',''))
                    outst = float(row[6].replace(',',''))
                    age = float(row[7])

                    if stock == 0:
                        if outst > 0:
                            style_cmds.append(('BACKGROUND', (5,row_idx), (5,row_idx), rl_colors.Color(0.7, 0.9, 0.7)))
                        else:
                            style_cmds.append(('BACKGROUND', (5,row_idx), (5,row_idx), rl_colors.Color(1, 0.7, 0.7)))
                    elif stock < 10:
                        style_cmds.append(('BACKGROUND', (5,row_idx), (5,row_idx), rl_colors.Color(1, 1, 0.7)))

                    if age > 365:
                        style_cmds.append(('TEXTCOLOR', (7,row_idx), (7,row_idx), rl_colors.red))
                        style_cmds.append(('FONTNAME', (7,row_idx), (7,row_idx), 'Helvetica-Bold'))
                except:
                    pass

            sku_table.setStyle(TableStyle(style_cmds))
            story.append(sku_table)

        def add_page_elements(canvas, doc):
            canvas.saveState()

            canvas.setFont('Helvetica-Bold', 10)
            canvas.setFillColor(rl_colors.HexColor('#667eea'))
            canvas.drawString(0.5*inch, A4[1] - 0.4*inch, f"Collection {self.curr_coll} Analysis")

            canvas.setFont('Helvetica', 8)
            canvas.setFillColor(rl_colors.grey)
            canvas.drawRightString(A4[0] - 0.5*inch, A4[1] - 0.4*inch,
                                  f"{datetime.now().strftime('%d %B %Y')}")

            canvas.setStrokeColor(rl_colors.HexColor('#667eea'))
            canvas.setLineWidth(1.5)
            canvas.line(0.5*inch, A4[1] - 0.5*inch, A4[0] - 0.5*inch, A4[1] - 0.5*inch)

            canvas.setFont('Helvetica', 7)
            canvas.setFillColor(rl_colors.grey)
            canvas.drawCentredString(A4[0]/2, 0.4*inch,
                                    "Professional Pattern & Color Analysis Report")
            canvas.drawCentredString(A4[0]/2, 0.25*inch,
                                    "Generated by Pattern Analytics Dashboard | Confidential Business Data")

            canvas.setFont('Helvetica-Bold', 8)
            canvas.drawRightString(A4[0] - 0.5*inch, 0.4*inch, f"Page {doc.page}")

            canvas.setStrokeColor(rl_colors.lightgrey)
            canvas.setLineWidth(0.5)
            canvas.line(0.5*inch, 0.6*inch, A4[0] - 0.5*inch, 0.6*inch)

            canvas.restoreState()

        try:
            doc.build(story, onFirstPage=add_page_elements, onLaterPages=add_page_elements)
            return str(fn)
        except Exception as e:
            print(f"❌ PDF generation error: {e}")
            return None

    def display(self):
        """Display dashboard"""
        h = """<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 45px; border-radius: 15px; margin-bottom: 25px; box-shadow: 0 12px 35px rgba(0,0,0,0.35);
        text-align: center; border: 3px solid #5568d3;'>
        <h1 style='color: #FFF; margin: 0; font-size: 46px; font-weight: 900; text-shadow: 3px 3px 6px rgba(0,0,0,0.4);'>
        🎨 Pattern & Color Dashboard</h1>
        <p style='color: #FFF; margin: 18px 0 0; font-size: 22px; font-weight: 700;'>
        ✅ Professional Edition</p></div>"""
        display(HTML(h))

        ctrl = widgets.VBox([
            widgets.HTML("<h3 style='color: #000; font-weight: 900;'>📋 Controls:</h3>"),
            self.coll_box, self.year_sel, self.btn_analyze,
            widgets.HTML("<div style='margin: 10px 0;'></div>"), self.btn_pdf
        ], layout=widgets.Layout(padding='22px', background_color='#f8f9fa',
                                border_radius='12px', border='2px solid #e0e0e0'))

        display(ctrl)
        display(self.out)


# ============================================================================
# USAGE
# ============================================================================

print("✅ Professional Pattern Dashboard Ready!")
print()
print("🚀 USAGE:")
print("dashboard = PatternColorDashboard(combined_df_updated)")
print("dashboard.display()")
print()
print("📁 PDF exports to: Desktop/pattern_color_analysis/YYYYMMDD_HHMMSS/")

✅ Professional Pattern Dashboard Ready!

🚀 USAGE:
dashboard = PatternColorDashboard(combined_df_updated)
dashboard.display()

📁 PDF exports to: Desktop/pattern_color_analysis/YYYYMMDD_HHMMSS/


In [5]:
# Do this:
results = run_pattern_analysis(combined_df)
combined_df_updated = results['df_updated']  # ← Updated!
dashboard = PatternColorDashboard(combined_df_updated)
# ← Use updated!

NameError: name 'run_pattern_analysis' is not defined

In [ ]:
dashboard.display()

In [7]:
# results = run_pattern_analysis(combined_df)
# combined_df_updated = results['df_updated']
# print(results['patterns'].head())

results.head()

AttributeError: 'dict' object has no attribute 'head'

In [9]:
# ============================================================================
# CELL 1: SAVE VARIABLES (ضعها في نهاية Notebook)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("💾 SAVING NOTEBOOK VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📋 STEP 1: List variables to save (Edit this list as needed)
# ═══════════════════════════════════════════════════════════════════════════

variables_to_save = [
    # Core DataFrames
    'combined_df', #the main dataframe
    'combined_df_updated', # updated dataframe for pattern dashboard
    'df',

    # Analysis Results
    'results_df', # for skus classification
    'classification_df',
    'forecasts_df',
    'inventory_df',
    'combined_df_updated',
    'sku_df',
    'section_df',

    # Dashboard Objects
    'dashboard',

    # Pattern Analysis
    'pattern_stats',
    'sku_level',
    'color_analysis',

    # Configs & Dicts
    'config',
    'metrics',

    # Add any other variables here
]

# ═══════════════════════════════════════════════════════════════════════════
# 📁 STEP 2: Create save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"
save_folder.mkdir(exist_ok=True)

# Create timestamped subfolder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
session_folder = save_folder / f"session_{timestamp}"
session_folder.mkdir(exist_ok=True)

print(f"\n📂 Save location: {session_folder}")
print()

# ═══════════════════════════════════════════════════════════════════════════
# 💾 STEP 3: Save each variable
# ═══════════════════════════════════════════════════════════════════════════

saved_vars = {}
success_count = 0
skip_count = 0
fail_count = 0

for var_name in variables_to_save:
    if var_name in globals():
        try:
            var_value = globals()[var_name]

            # Save to pickle
            file_path = session_folder / f"{var_name}.pkl"
            with open(file_path, 'wb') as f:
                pickle.dump(var_value, f)

            # Get file size
            size_kb = file_path.stat().st_size / 1024
            size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"

            saved_vars[var_name] = {
                'saved': True,
                'type': type(var_value).__name__,
                'size': size_str
            }

            print(f"✅ {var_name:<25} | {saved_vars[var_name]['type']:<15} | {size_str}")
            success_count += 1

        except Exception as e:
            saved_vars[var_name] = {'saved': False, 'error': str(e)}
            print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
            fail_count += 1
    else:
        print(f"⚠️  {var_name:<25} | NOT FOUND (skipped)")
        skip_count += 1

# ═══════════════════════════════════════════════════════════════════════════
# 📝 STEP 4: Save metadata
# ═══════════════════════════════════════════════════════════════════════════

metadata = {
    'timestamp': timestamp,
    'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'saved_variables': saved_vars,
    'notebook_name': 'Current Notebook',  # You can customize this
    'success_count': success_count,
    'skip_count': skip_count,
    'fail_count': fail_count
}

metadata_path = session_folder / "metadata.pkl"
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

# ═══════════════════════════════════════════════════════════════════════════
# ✅ STEP 5: Summary
# ═══════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("✅ SAVE COMPLETE!")
print("=" * 80)
print(f"📂 Location: {session_folder}")
print(f"✅ Saved:    {success_count} variables")
print(f"⚠️  Skipped:  {skip_count} variables (not found)")
print(f"❌ Failed:   {fail_count} variables")
print("=" * 80)
print()
print("💡 To load these variables in another notebook, use CELL 2")
print("=" * 80)

💾 SAVING NOTEBOOK VARIABLES

📂 Save location: C:\Users\User\Desktop\notebook_variables\session_20260118_151536

✅ combined_df               | DataFrame       | 1592.6 MB
✅ combined_df_updated       | DataFrame       | 1592.6 MB
✅ df                        | DataFrame       | 1454.8 MB
⚠️  results_df                | NOT FOUND (skipped)
⚠️  classification_df         | NOT FOUND (skipped)
⚠️  forecasts_df              | NOT FOUND (skipped)
✅ inventory_df              | DataFrame       | 17.6 MB
✅ combined_df_updated       | DataFrame       | 1592.6 MB
✅ sku_df                    | DataFrame       | 15.0 MB
✅ section_df                | DataFrame       | 855.7 KB
❌ dashboard                 | FAILED: Can't pickle <built-in function input>: 
⚠️  pattern_stats             | NOT FOUND (skipped)
⚠️  sku_level                 | NOT FOUND (skipped)
⚠️  color_analysis            | NOT FOUND (skipped)
⚠️  config                    | NOT FOUND (skipped)
⚠️  metrics                   | NOT FOUND (s